# Sandogh FAQ chatbot (RAG)

A retrieval-augmented FAQ assistant for the Iran National Innovation Fund (صندوق نوآوری و شکوفایی).

Pipeline:
1. Load the scraped Q&A pairs from `faq_qa_pairs.csv` and turn each row into a text chunk.
2. Store the chunks in an embedded Weaviate vector database. Weaviate embeds them through the local Flask server in `flask_app.py` (model: `BAAI/bge-m3`).
3. For a user question, retrieve the most similar chunks.
4. Send the question and the retrieved chunks to Claude, which answers using only that context.

Requirements: `pip install anthropic weaviate-client sentence-transformers flask tqdm`, and an `ANTHROPIC_API_KEY` in the environment.

## 1. Imports
Importing `flask_app` starts the local embedding server and waits until it is ready.

In [ ]:
import csv
import os
from typing import Any, Dict, List

import tqdm
import weaviate
from weaviate.classes.config import Configure, DataType, Property
from weaviate.util import generate_uuid5

import flask_app  # starts the local embedding server on port 5050
from utils import generate_with_single_input

## 2. Load the FAQ data and build chunks
Each CSV row becomes one chunk such as `category: ... | question: ... | answer: ...`. The original columns are kept as separate properties too.

In [ ]:
def get_chunks_from_csv(file_path: str) -> List[Dict[str, Any]]:
    """Read the CSV and return one dict per row: the combined 'chunk' text plus the original columns."""
    chunks = []

    with open(file_path, "r", encoding="utf-8") as f:
        source_csv = csv.DictReader(f)

        for row in source_csv:
            # Join all non-empty columns into a single text that will be embedded.
            chunk_text = " | ".join(
                f"{column}: {row[column]}"
                for column in source_csv.fieldnames
                if row[column]  # skip empty columns
            )
            chunks.append({"chunk": chunk_text, **row})
    return chunks

In [ ]:
created_chunks = get_chunks_from_csv("faq_qa_pairs.csv")
print(f"Loaded {len(created_chunks)} chunks")

## 3. Start the embedded Weaviate database
Data is saved on disk in `./chatbot` (or under `$COLLECTION_M3` if that variable is set), so it survives restarts. The `text2vec-transformers` module is pointed at our Flask server.

In [ ]:
collection_base = os.getenv('COLLECTION_M3', './')
persistence_path = os.path.abspath(os.path.join(collection_base, 'chatbot'))

# URL of the local embedding server started by `import flask_app`.
INFERENCE_URL = f"http://{flask_app.HOST}:{flask_app.PORT}"

client = weaviate.connect_to_embedded(
    port=8079,
    grpc_port=50050,
    persistence_data_path=persistence_path,
    environment_variables={
        "ENABLE_API_BASED_MODULES": "true",
        "ENABLE_MODULES": "text2vec-transformers",
        "TRANSFORMERS_INFERENCE_API": INFERENCE_URL,
    }
)

## 4. Create the collection (first run only)
Only the `chunk` property is embedded. `question`, `answer` and `category` are stored as plain text.

In [ ]:
COLLECTION_NAME = "sandogh_faq"

if not client.collections.exists(COLLECTION_NAME):
    collection = client.collections.create(
        name=COLLECTION_NAME,
        vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
            name="vector",               # name used to access the vectors of the objects
            source_properties=["chunk"], # only the combined chunk text is embedded
            vectorize_collection_name=False,
            inference_url=INFERENCE_URL, # our Flask embedding server
        )],
        properties=[
            Property(name="chunk", data_type=DataType.TEXT),
            Property(name="question", data_type=DataType.TEXT),
            Property(name="answer", data_type=DataType.TEXT),
            Property(name="category", data_type=DataType.TEXT),
        ]
    )
else:
    # Reuse the collection saved on disk by a previous run.
    collection = client.collections.get(COLLECTION_NAME)

## 5. Insert the chunks
Each object's ID is derived from its content (`generate_uuid5`), so inserting the same chunk again overwrites it instead of creating a duplicate. This makes it safe to re-run the insert whenever the collection holds fewer objects than the CSV, e.g. after an interrupted first load.

In [ ]:
if collection.aggregate.over_all().total_count < len(created_chunks):
    # Weaviate calls the embedding server for each object while batching.
    with collection.batch.fixed_size(batch_size=32, concurrent_requests=4) as batch:
        for chunk_obj in tqdm.tqdm(created_chunks):
            batch.add_object(properties=chunk_obj, uuid=generate_uuid5(chunk_obj))

    if collection.batch.failed_objects:
        print(f"Failed to insert {len(collection.batch.failed_objects)} objects. First error:")
        print(collection.batch.failed_objects[0].message)

print(f"Total count: {collection.aggregate.over_all().total_count}")

## 6. Retrieval and answer generation
`retrieve` finds the closest FAQ chunks with vector search. `answer_question` sends them to Claude as context.

In [ ]:
# Instructions for Claude: answer only from the retrieved FAQ text, in the user's language.
SYSTEM_PROMPT = """You are the FAQ assistant of the Iran National Innovation Fund (صندوق نوآوری و شکوفایی).
Answer the user's question using only the information in the provided FAQ context.
If the answer is not in the context, say that you don't know.
Answer in the same language as the user's question."""

# The retrieved chunks and the question are wrapped in XML tags so Claude can tell them apart.
PROMPT = """<context>
{context}
</context>

<question>
{question}
</question>"""


def retrieve(question: str, top_k: int = 3) -> List[str]:
    """Return the text of the top_k chunks most similar to the question."""
    # near_text embeds the question through the same Flask server, then searches by vector similarity.
    response = collection.query.near_text(question, limit=top_k)
    return [obj.properties["chunk"] for obj in response.objects]


def answer_question(question: str, top_k: int = 3) -> str:
    """Retrieve context for the question and ask Claude to answer from it."""
    context = "\n\n".join(retrieve(question, top_k))
    prompt = PROMPT.format(question=question, context=context)
    return generate_with_single_input(prompt, system=SYSTEM_PROMPT)["content"]

## 7. Try it
First look at what retrieval returns, then ask Claude for the answer.

In [ ]:
user_question = "شرایط دریافت ضمانت‌نامه چیست؟"  # "What are the requirements for getting a guarantee?"

for i, chunk in enumerate(retrieve(user_question), start=1):
    print(f"===== Result {i} =====")
    print(chunk)
    print()

In [ ]:
print("Answer:")
print(answer_question(user_question))

## 8. Shut down
Closes the connection and stops the embedded Weaviate process. The data stays on disk.

In [ ]:
client.close()